# Full Sample Recovery — ESG DART (memberC)

**Goal**: bridge user pipeline (N=243, fy 2021-2023, 81 firms) to MemberA universe (N=381, fy 2022-2024, 127 firms) **without fabrication**.

**Output sets**:
- `firm_year_lineage.csv` — 556 rows = 381 MemberA universe ∪ 175 user-native extension
- `failure_taxonomy.csv` — every missing/dropped firm-year classified with reason
- `merged_full_sample.csv` — lineage joined with features (where available)
- `recovery_audit_report.md`, `robustness_comparison.md`, `qa_findings.md`

**Robustness layer**:
- preprocessing comparison (exp_B/E/F): `preprocessing_comparison.csv`
- pilot vs full sign reversal (N=30 resamples × 200): `pilot_vs_full_sign_reversal.csv`
- verbosity orthogonalization: `verbosity_diagnostics.csv`
- tokenizer diagnostics: `tokenizer_diagnostics.csv`

## Decision Box — Universe choice
**Alternative**: (A) MemberA 381 only / (B) User 243 only / (C) Union — both as separate views.
**Choice**: (C). Preserves user's pipeline as `user_native_extension` (175 OOU firm-years) alongside MemberA target (60 SUCCESS / 313 awaiting refetch / 8 partial).
**Justification**: lineage integrity is paramount. The user pipeline is calibrated to esg_year=fiscal_year+1 (different timing rule). Both views answer the same research question.
**Limitation**: features were not regenerated under MemberA timing rule for OOU rows.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

OUT = Path('.')  # outputs/recovery dir

# 1. Coverage audit (8-stage funnel)
audit = pd.read_csv(OUT/'coverage_audit.csv')
print('[STEP 1 — Coverage Audit]')
print(audit.to_string(index=False))

In [ ]:
# 2. Failure taxonomy
tax = pd.read_csv(OUT/'failure_taxonomy.csv', dtype={'stock_code': str})
tax['stock_code'] = tax['stock_code'].str.zfill(6)
combo = tax.apply(lambda r: r['status'] if pd.isna(r.get('sub_status')) or r['sub_status']=='' else r['sub_status'], axis=1)
print('[STEP 2 — Failure Taxonomy]')
print(combo.value_counts().to_string())

In [ ]:
# 3. Lineage table
lin = pd.read_csv(OUT/'firm_year_lineage.csv', dtype={'stock_code':str})
lin['stock_code'] = lin['stock_code'].str.zfill(6)
print('[STEP 3 — Lineage by universe × source_pipeline]')
print(lin.groupby(['universe','source_pipeline']).size().to_string())

In [ ]:
# 4. Recovery plan
rs = pd.read_csv(OUT/'recovery_summary_table.csv')
print('[STEP 4 — Recovery Plan]')
print(rs.to_string(index=False))

In [ ]:
# 5A — Preprocessing robustness
prep = pd.read_csv(OUT/'preprocessing_comparison_wide.csv', index_col=0)
print('[STEP 5A — exp_B vs exp_E vs exp_F]')
print(prep.to_string())

In [ ]:
# 5B — Tokenizer diagnostics
tok = pd.read_csv(OUT/'tokenizer_diagnostics.csv')
print('[STEP 5B — Tokenizer]')
print(tok.to_string(index=False))

In [ ]:
# 5C — Pilot vs full sign reversal
sr = pd.read_csv(OUT/'pilot_vs_full_sign_reversal.csv')
print('[STEP 5C — N=30 resample reversal rate]')
print(sr.to_string(index=False))

In [ ]:
# 5D — Verbosity diagnostics
verb = pd.read_csv(OUT/'verbosity_diagnostics.csv')
print('[STEP 5D — Orthogonalize vs log total_tokens]')
print(verb.to_string(index=False))

In [ ]:
# 7 — QA assertions
qa = pd.read_csv(OUT/'qa_assertions.csv')
print('[STEP 7 — QA Assertions]')
print(qa.to_string(index=False))
print(f'\nPASS: {(qa["status"]=="PASS").sum()}/{len(qa)}')

In [ ]:
# 8 — Final merged_full_sample
mg = pd.read_csv(OUT/'merged_full_sample.csv', dtype={'stock_code':str})
mg['stock_code'] = mg['stock_code'].str.zfill(6)
print(f'[STEP 8 — merged_full_sample shape: {mg.shape}]')
print('\nBy status × features availability:')
print(mg.groupby('status').agg(n=('stock_code','count'),
                                  feat_present=('total_tokens', lambda s: s.notna().sum())).to_string())

## To proceed to N=381

Run `refetch_dart_api.py` with `OPENDART_API_KEY` set.
After ~5 min runtime (~626 API calls), 313 missing zips will land in `recovered_zips/`.
Re-run section extraction → passage filter → preprocessing → feature_builder on the union of existing + recovered zips.

Expected final state: ~373 SUCCESS / 8 partial (financial-holding) / 0 unrecoverable.

## Honest gap statement
Items still open per COMPARISON_30단계_4인.html:
- #10 head-to-head Kiwi vs Okt sensitivity run
- #15–18 expanded dictionary + fastText + θ sweep + candidate review log

These are dictionary-expansion methods that need corpus regeneration, not recovery. Documented as next-priority future work.